<a href="https://colab.research.google.com/github/profhannahchang/profhannahchang.github.io/blob/main/tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction

**Overview**

This is a tutorial on text analysis for behavioral science research, written by Sudeep Bhatia (bhatiasu@sas.upenn.edu). There will be two parts to the tutorial. In the first, we will try out some simple text analysis operations. In the second, we will perform an automated content analysis on a text dataset.

The data for this tutorial involves 5,000 movie synopses scraped from IMDB.com.

**Files**

I have all files necessary for the tutorial publically available on my Google Drive tutorial folder (https://drive.google.com/drive/folders/1fAW_CdxJ_MhC-w4kWzr7B9nfUz4KYHHj?usp=sharing). Here are the links and descriptions:
*   "coding_data.csv" --> main dataset for automated coding
*   "coding_dictionary_sentiment.csv" --> Dictionary of words with valence ratings obtained from Wilson et al. (2005) and available at http://mpqa.cs.pitt.edu/lexicons/subj_lexicon/
*    "coding_dictionary_pronouns.csv" --> A list of male and female pronouns.
*    "coding_dictionary_other.csv" --> A list of various constructs.
*    "glove.6B.300d.txt" --> GLOVE semantic space, obtained from https://nlp.stanford.edu/projects/glove/.

You can use the code in this tutorial to replicate all of these tests on your own datasets and dictionaries if you make sure that the files are in the same format.

**Additional resources**
*   Humphreys & Wang (2018) and Garten et al. (2018) provide an overview of how automated coding can be used for text analysis in the behavioral sciences.
*   Dan Jurafsky and James Martin have a popular free textbook on natural language processing, available at: https://web.stanford.edu/~jurafsky/slp3/.
*   NLTK is a easy-to-use Python toolkit for doing natural language processing. It comes with a textbook. Both are available here: https://www.nltk.org/.


**References**

*   Garten, J., Hoover, J., Johnson, K. M., Boghrati, R., Iskiwitch, C., & Dehghani, M. (2018). Dictionaries and distributions: Combining expert knowledge and large scale textual data content analysis. Behavior research methods, 50(1), 344-361.
*   Humphreys, A., & Wang, R. J. H. (2018). Automated text analysis for consumer research. Journal of Consumer Research, 44(6), 1274-1306.
*   Wilson, T., Wiebe, J., & Hoffmann, P. (2005). Recognizing contextual polarity in phrase-level sentiment analysis. In Proceedings of human language technology conference and conference on empirical methods in natural language processing (pp. 347-354).





# Simple Text Analysis

Here we will try out some simple text analysis to give you a feel for how text is analyzed in a language like Python.

But before we begin analyzing text, let's do a quick arithmetic operation.

In [ ]:
x = 3
y = -2*x

We can print "y" to see what its value is.

In [ ]:
print(y)

That was easy.

Ok, now let's create a new variable called "text", which is a string of characters from The Princess Bride.

In [ ]:
text = 'Hello. My name is Inigo Montoya. You killed my father. Prepare to die.'

We can print "text" to see what it contains.

In [ ]:
print(text)

Now I will perform an operation on "text", to generate a new variable called "processed_text". Try to guess what each this operation does.

In [ ]:
processed_text = text.lower()
print(processed_text)

What about this operation?

In [ ]:
count_to = text.count('to')
print(count_to)

This one?

In [ ]:
processed_text = text.lower().replace('.',' ').split(' ')
print(processed_text)

Here are some other simple operations that you can perform on text.

In [ ]:
text = 'Hello. My name is Inigo Montoya. You killed my father. Prepare to die.'

first_char = text[4]
char_ten_to_twenty = text[10:20]
sentences = text.split('.')
second_sentence = text.split('.')[1]

print(second_sentence)

We can also use functions to automatically implement a set of operations for us. Here we are defining a function that automaticaly converts a string to a list of words.

In [ ]:
def wordlist(input_text):  #create a function that will automatically convert a string to a list of words
    text2 = input_text.replace('.',' ') #remove periods from string (i.e. replace them with white space)
    text3 = text2.lower() #lower case string
    text4 = text3.split(' ')  #split up string based on white space to obtain word list
    return text4 #return word list

In [ ]:
print(wordlist('my name is sudeep'))

We can call that function with "text" as the input and save its output as "processed_text"

In [ ]:
text = 'Hello. My is Inigo Montoya. You killed my father. Prepare to die.'
processed_text = wordlist(text) #call wordlist function with text
print(processed_text.count('to'))
print(processed_text)

# Set Up for Coding
First mount your Google Drive.

In [ ]:
#mount google drive
from google.colab import drive
drive.mount('/content/drive/')


Now import the Python modules that we will be using in our code.

In [ ]:
#import necessary modules
import pandas, numpy, csv, string
from scipy.spatial import distance
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn import preprocessing

Now declare the function that we will use for doing a word count dictionary analysis on the text data.

Here "coding_data" is a dataframe with all the data and "dictionary" is a dataframe with dictionary words and values.

The function will output "coding_data" with the coding dimensions as new columns.



In [ ]:
#function for doing a word count dictionary analysis on text data
def coding_function_word_count(coding_data,dictionary):

  #convert text column to a list of strings
  text_data = list(coding_data['Text'])

  #extract coding dimensions
  coding_dimensions = list(dictionary.columns)[1:]

  #iterate over coding dimensions
  for dimension in coding_dimensions:

    #print that you are starting dimension
    print("Starting",dimension)

    #create dictionary for dimension
    dimension_dict = {}
    for index, row in dictionary.iterrows():
      dimension_dict[row['Word']] = row[dimension]

    #initiate text_value variable
    text_value = []

    #iterate over texts
    for text in text_data:

      #remove punctuation, lower case, and split by whitespace
      clean_text = str(text).translate(str.maketrans('', '', string.punctuation)).lower().split(' ')

      #initiate word)count and sum_value variables
      word_count = 0
      sum_value = 0

      #add up values for words in our dictionary
      for word in clean_text:
          word_count = word_count + 1
          if  word in dimension_dict:
              sum_value = sum_value + float(dimension_dict[word])

      #get average value in text
      if word_count > 0:
        average_value = float(sum_value)/float(word_count)
      else:
        average_value = -99

      #append average value to text_value
      text_value.append(average_value)

      #print number of processed texts
      if len(text_value)%25000 == 0:
        print("Processed", len(text_value),"texts")

    #add text_value to initial dataframe
    coding_data[dimension] = text_value

    #print that you have completed dimension
    print("Completed",dimension)

  return coding_data

Now declare the function that we will use for doing a semantic space dictionary analysis on the text data.

Here "coding_data" is a dataframe with all the data, and "dictionary" is a dataframe with dictionary words and values.

The function will output "coding_data" with the coding dimensions as new columns.

Note that this function only works for dictionaries that have values of "1" and "0" (it does not work for continuously valued coding dimensions).

Finally, before loading function we have to load the glove vector dictionary. This will take some time.



In [ ]:
#load glove vectors
f = open("/content/drive/My Drive/Tutorial/glove.6B.300d.txt","r")
glove300 = {}
for line in f:
    line2 = line.split(' ')
    line3 =  [float(l) for l in line2[1:len(line2)]]
    glove300[line2[0]] = line3

#function for doing a semantic dictionary analysis on text data
def coding_function_semantic_space(coding_data,dictionary):

  #convert text column to a list of strings
  text_data = list(coding_data['Text'])

  #extract coding dimensions
  coding_dimensions = list(dictionary.columns)[1:]

  #iterate over coding dimensions
  for dimension in coding_dimensions:

    #print that you are starting dimension
    print("Starting",dimension)

    #create vectors for dimension
    dimension_vector = numpy.zeros(300)
    word_count = 0
    for index, row in dictionary.iterrows():
      if row[dimension] == 1 and row['Word'] in glove300:
         dimension_vector = dimension_vector + numpy.asarray(glove300[row['Word']])
         word_count= word_count + 1
    dimension_vector = numpy.divide(dimension_vector,word_count)

    #initiate text_value variable
    text_value = []

    #iterate over texts
    for text in text_data:

      #remove punctuation, lower case, and split by whitespace
      clean_text = str(text).translate(str.maketrans('', '', string.punctuation)).lower().split(' ')

      #initiate wordcount and text_vector variables
      text_vector = numpy.zeros(300)
      word_count = 0

      #add up vectors for word in text
      for word in clean_text:
          if word in glove300:
            text_vector = text_vector  + numpy.asarray(glove300[word])
            word_count = word_count + 1

      #get similarity of text vector with dimension_vector
      if word_count > 0:
        text_vector = numpy.divide(text_vector,word_count)
        similarity = 1 - distance.cosine(text_vector,dimension_vector)
      else:
        similarity = -99

      #append average value to text_value
      text_value.append(similarity)

      #print number of processed texts
      if len(text_value)%10000 == 0:
        print("Processed", len(text_value),"texts")

    #add text_value to initial dataframe
    coding_data[dimension] = text_value

    #print that you have completed dimension
    print("Completed",dimension)

  return coding_data

# Automated Coding of Text

We start by uploading the data that we will be coding. In our case, this is stored as "coding_data.csv". You can change the file path and file name in the code, if you decide to use your own dataset. We will be uploading it as a pandas dataframe. After uploading, we will print out some summary statistics for the dataset.

Note that the code used here assumes that the column with the text data is called "Text".



In [ ]:
#upload coding data as pandas dataframe
coding_data = pandas.read_csv('/content/drive/My Drive/Tutorial/coding_data.csv', encoding = "ISO-8859-1")

Inspect dataset.

In [ ]:
#inspect dataset
coding_data

Print out some summary stats.

In [ ]:
#print out summary stats
print("Number of observations:",len(coding_data))
print("Averge number of characters in text:", numpy.mean([len(text) for text in coding_data['Text']]))

Now upload dictionary, perform word count-based coding on each of the dictionary columns, and save as "coding_data_processed" into Google Drive.

In [ ]:
#upload dictionary
dictionary = pandas.read_csv('/content/drive/My Drive/Tutorial/coding_dictionary_other.csv', encoding = "ISO-8859-1")

#do coding
coding_data_processed = coding_function_word_count(coding_data,dictionary)

#print summary mean and se of each dimension in dictionary
for dimension in list(dictionary.columns)[1:]:
  print(dimension,coding_data_processed[dimension].mean(), '+-',coding_data_processed[dimension].std()/(coding_data_processed[dimension].count()**.5))

#save
coding_data_processed.to_csv('/content/drive/My Drive/Tutorial/coding_data_processed.csv')


We can repeat the above analysis using our semantic space code.

In [ ]:
#upload dictionary
dictionary = pandas.read_csv('/content/drive/My Drive/Tutorial/coding_dictionary_other.csv', encoding = "ISO-8859-1")

#do coding on random sample of 10000 texts
coding_data_processed = coding_function_semantic_space(coding_data,dictionary)

#print summary mean, stdev, and count of each dimension in dictionary
for dimension in list(dictionary.columns)[1:]:
  print(dimension,coding_data_processed[dimension].mean(), '+-',coding_data_processed[dimension].std()/(coding_data_processed[dimension].count()**.5))

#save
coding_data_processed.to_csv('/content/drive/My Drive/Tutorial/coding_data_processed.csv')
